#CELL 1
# Workforce360 — KPI Validation

This notebook validates the executive workforce KPIs that will later power the Streamlit dashboard and Power BI reporting layer.

## KPI Definitions

| KPI | Definition in Workforce360 |
|---|---|
| Total Employees | Number of employee records in the dataset |
| Active Employees | Records where `attrition_flag = 0` |
| Attrition Rate | Mean of `attrition_flag` × 100 |
| Average Monthly Salary | Mean of `monthly_income` |
| Average Tenure | Mean of `years_at_company` |
| Average Job Satisfaction | Mean of `job_satisfaction` |
| Recent Promotion Rate | Percentage where `years_since_promotion <= 2` |
| Overtime Percentage | Mean of `overtime_flag` × 100 |

> **Important:** The IBM dataset is a historical portfolio dataset. “Active Employees” and “Recent Promotion Rate” are analytical proxy definitions, not live HRIS measures.

In [1]:
#CELL 2
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.analytics.kpis import workforce_kpi_summary
from src.data.load import load_raw_ibm_hr
from src.data.preprocess import clean_hr_data

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

In [2]:
#CELL 3
raw_df = load_raw_ibm_hr()
df = clean_hr_data(raw_df)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df[
    [
        "employee_id",
        "department",
        "monthly_income",
        "years_at_company",
        "attrition_flag",
        "overtime_flag",
        "years_since_promotion",
    ]
].head()

Rows: 1470
Columns: 43


,employee_id,department,monthly_income,years_at_company,attrition_flag,overtime_flag,years_since_promotion
0,EMP0001,Sales,5993,6,1,1,0
1,EMP0002,Research & Development,5130,10,0,0,1
2,EMP0004,Research & Development,2090,0,1,1,0
3,EMP0005,Research & Development,2909,8,0,1,3
4,EMP0007,Research & Development,3468,2,0,0,2


In [3]:
#CELL 4
kpis = workforce_kpi_summary(df)

kpi_table = pd.DataFrame(
    {
        "KPI": [
            "Total Employees",
            "Active Employees",
            "Attrition Rate (%)",
            "Average Monthly Salary",
            "Average Tenure (Years)",
            "Average Job Satisfaction",
            "Recent Promotion Rate (%)",
            "Overtime Percentage (%)",
        ],
        "Value": [
            kpis["total_employees"],
            kpis["active_employees"],
            kpis["attrition_rate"],
            kpis["average_monthly_salary"],
            kpis["average_tenure_years"],
            kpis["average_job_satisfaction"],
            kpis["recent_promotion_rate"],
            kpis["overtime_percentage"],
        ],
    }
)

kpi_table

,KPI,Value
0,Total Employees,"1,470.00"
1,Active Employees,"1,233.00"
2,Attrition Rate (%),16.12
3,Average Monthly Salary,"6,502.93"
4,Average Tenure (Years),7.01
5,Average Job Satisfaction,2.73
6,Recent Promotion Rate (%),74.63
7,Overtime Percentage (%),28.30


In [4]:
#CELL 5
manual_total_employees = len(df)
manual_active_employees = int((df["attrition_flag"] == 0).sum())
manual_attrition_rate = round(df["attrition_flag"].mean() * 100, 2)
manual_average_salary = round(df["monthly_income"].mean(), 2)
manual_average_tenure = round(df["years_at_company"].mean(), 2)
manual_job_satisfaction = round(df["job_satisfaction"].mean(), 2)
manual_promotion_rate = round(
    (df["years_since_promotion"] <= 2).mean() * 100,
    2,
)
manual_overtime_percentage = round(df["overtime_flag"].mean() * 100, 2)

validation_table = pd.DataFrame(
    {
        "KPI": [
            "Total Employees",
            "Active Employees",
            "Attrition Rate (%)",
            "Average Monthly Salary",
            "Average Tenure (Years)",
            "Average Job Satisfaction",
            "Recent Promotion Rate (%)",
            "Overtime Percentage (%)",
        ],
        "KPI Function": [
            kpis["total_employees"],
            kpis["active_employees"],
            kpis["attrition_rate"],
            kpis["average_monthly_salary"],
            kpis["average_tenure_years"],
            kpis["average_job_satisfaction"],
            kpis["recent_promotion_rate"],
            kpis["overtime_percentage"],
        ],
        "Manual Calculation": [
            manual_total_employees,
            manual_active_employees,
            manual_attrition_rate,
            manual_average_salary,
            manual_average_tenure,
            manual_job_satisfaction,
            manual_promotion_rate,
            manual_overtime_percentage,
        ],
    }
)

validation_table["Matches"] = (
    validation_table["KPI Function"] == validation_table["Manual Calculation"]
)

validation_table

,KPI,KPI Function,Manual Calculation,Matches
0,Total Employees,"1,470.00","1,470.00",True
1,Active Employees,"1,233.00","1,233.00",True
2,Attrition Rate (%),16.12,16.12,True
3,Average Monthly Salary,"6,502.93","6,502.93",True
4,Average Tenure (Years),7.01,7.01,True
5,Average Job Satisfaction,2.73,2.73,True
6,Recent Promotion Rate (%),74.63,74.63,True
7,Overtime Percentage (%),28.30,28.30,True


In [5]:
#CELL 6
department_kpis = (
    df.groupby("department", observed=False)
    .agg(
        employees=("employee_id", "count"),
        attrition_rate=("attrition_flag", "mean"),
        average_monthly_income=("monthly_income", "mean"),
        average_tenure_years=("years_at_company", "mean"),
        overtime_percentage=("overtime_flag", "mean"),
        average_job_satisfaction=("job_satisfaction", "mean"),
    )
    .reset_index()
)

department_kpis["attrition_rate"] = (
    department_kpis["attrition_rate"] * 100
).round(2)

department_kpis["overtime_percentage"] = (
    department_kpis["overtime_percentage"] * 100
).round(2)

department_kpis["average_monthly_income"] = (
    department_kpis["average_monthly_income"].round(2)
)

department_kpis["average_tenure_years"] = (
    department_kpis["average_tenure_years"].round(2)
)

department_kpis["average_job_satisfaction"] = (
    department_kpis["average_job_satisfaction"].round(2)
)

department_kpis.sort_values(
    by="attrition_rate",
    ascending=False,
).reset_index(drop=True)

,department,employees,attrition_rate,average_monthly_income,average_tenure_years,overtime_percentage,average_job_satisfaction
0,Sales,446,20.63,"6,959.17",7.28,28.70,2.75
1,Human Resources,63,19.05,"6,654.51",7.24,26.98,2.60
2,Research & Development,961,13.84,"6,281.25",6.86,28.20,2.73


#CELL 7
## KPI Validation Notes

- All KPI function outputs matched their independent manual calculations.
- Total employee records represent the 1,470 observations in the IBM HR portfolio dataset.
- “Active Employees” is defined as records with `attrition_flag = 0`; this is an analytical proxy because the source data is historical.
- “Recent Promotion Rate” is defined as employees with `years_since_promotion <= 2`; it is a proxy because the source dataset does not provide a direct promotion-event history.
- Department-level metrics should be interpreted as descriptive patterns in this dataset, not causal explanations or real organizational performance measures.